# Task 4.1: Experimento de apoyo · máscara causal

Para la pregunta a. se entrena el mismo Mini-GPT dos veces por pocos pasos: una con la máscara causal y otra sin ella (atención bidireccional).

## Imports y reproducibilidad

In [1]:
import os
import random

import numpy as np
import torch

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Modelo

In [2]:
import math
import time

import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * gamma + beta


def make_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.WQ = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WK = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WV = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WO = nn.Parameter(torch.randn(d_model, d_model) * 0.02)

        self.gamma1 = nn.Parameter(torch.ones(d_model))
        self.beta1 = nn.Parameter(torch.zeros(d_model))

        self.W1 = nn.Parameter(torch.randn(d_model, d_ff) * 0.02)
        self.bias1 = nn.Parameter(torch.zeros(d_ff))
        self.W2 = nn.Parameter(torch.randn(d_ff, d_model) * 0.02)
        self.bias2 = nn.Parameter(torch.zeros(d_model))

        self.gamma2 = nn.Parameter(torch.ones(d_model))
        self.beta2 = nn.Parameter(torch.zeros(d_model))

    def masked_self_attention(self, x, causal_mask):
        B, T, C = x.shape
        Q = (x @ self.WQ).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = (x @ self.WK).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = (x @ self.WV).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, n_heads, T, T)
        scores = scores.masked_fill(causal_mask[:T, :T], float("-inf"))
        attn_w = F.softmax(scores, dim=-1)

        out = (attn_w @ V).transpose(1, 2).contiguous().view(B, T, C)
        return out @ self.WO

    def feed_forward(self, x):
        h = F.relu(x @ self.W1 + self.bias1)
        return h @ self.W2 + self.bias2

    def forward(self, x, causal_mask):
        # pre-LayerNorm (estilo GPT-2): estabiliza el entrenamiento al apilar capas
        x = x + self.masked_self_attention(layer_norm(x, self.gamma1, self.beta1), causal_mask)
        x = x + self.feed_forward(layer_norm(x, self.gamma2, self.beta2))
        return x


class MiniGPT(nn.Module):
    def __init__(self, V, d_model, n_heads, n_layers, block_size):
        super().__init__()
        self.E = nn.Parameter(torch.randn(V, d_model) * 0.02)
        self.register_buffer("PE", make_positional_encoding(block_size, d_model))
        self.register_buffer("causal_mask", torch.triu(torch.ones(block_size, block_size), diagonal=1).bool())

        self.blocks = nn.ModuleList([Block(d_model, n_heads, 4 * d_model) for _ in range(n_layers)])

        self.gamma_f = nn.Parameter(torch.ones(d_model))
        self.beta_f = nn.Parameter(torch.zeros(d_model))
        self.Wlm = nn.Parameter(torch.randn(d_model, V) * 0.02)
        self.blm = nn.Parameter(torch.zeros(V))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.E[idx] + self.PE[:T]
        for block in self.blocks:
            x = block(x, self.causal_mask)
        x = layer_norm(x, self.gamma_f, self.beta_f)
        logits = x @ self.Wlm + self.blm  # (B, T, V)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## Datos

In [3]:
import ssl
import urllib.request

import certifi

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = os.path.join("data", "input.txt")
if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    ctx = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(URL, context=ctx) as r, open(DATA_PATH, "wb") as f:
        f.write(r.read())

with open(DATA_PATH, encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

# mismos hiperparámetros que el Task 1
config = dict(block_size=128, d_model=128, n_heads=4, n_layers=3)
block_size = config["block_size"]
batch_size, lr = 64, 3e-4

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(d):
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## ¿La posición t ve el futuro?

Prueba directa, sin entrenar: se cambian los caracteres posteriores a la posición t y se mide cuánto cambian los logits en t.

In [4]:
torch.manual_seed(SEED)
probe = MiniGPT(vocab_size, **config).to(device).eval()
x = torch.tensor([encode("ROMEO:\nBut soft, what light through yonder window breaks?")], device=device)
t = 10
x_future = x.clone()
x_future[0, t + 1:] = torch.randint(vocab_size, (x.shape[1] - t - 1,))  # solo cambia el futuro

with torch.no_grad():
    for name, mask in (("causal", probe.causal_mask.clone()), ("bidireccional", torch.zeros_like(probe.causal_mask))):
        probe.causal_mask.copy_(mask)
        diff = (probe(x)[0][0, t] - probe(x_future)[0][0, t]).abs().max().item()
        print(f"{name:14} cambio máximo en los logits de la posición {t}: {diff:.2e}")

causal         cambio máximo en los logits de la posición 10: 0.00e+00
bidireccional  cambio máximo en los logits de la posición 10: 5.92e-04


## Entrenamiento con y sin máscara

In [5]:
STEPS = 700


@torch.no_grad()
def eval_loss(model, d, n_batches=20):
    model.eval()
    g = torch.Generator().manual_seed(0)
    losses = []
    for _ in range(n_batches):
        ix = torch.randint(len(d) - block_size - 1, (batch_size,), generator=g)
        x = torch.stack([d[i:i + block_size] for i in ix]).to(device)
        y = torch.stack([d[i + 1:i + block_size + 1] for i in ix]).to(device)
        losses.append(model(x, y)[1].item())
    model.train()
    return sum(losses) / len(losses)


def train(causal):
    torch.manual_seed(SEED)
    model = MiniGPT(vocab_size, **config).to(device)
    if not causal:
        # máscara vacía: la posición t puede atender a t+1, ..., T (atención bidireccional)
        model.causal_mask.zero_()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(1, STEPS + 1):
        xb, yb = get_batch(train_data)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % 100 == 0:
            print(f"{'causal' if causal else 'bidireccional':14} paso {step:4d} | loss {loss.item():.4f}")
    return model


models = {"causal": train(True), "bidireccional": train(False)}
for name, m in models.items():
    print(f"{name:14} train {eval_loss(m, train_data):.4f} | val {eval_loss(m, val_data):.4f}")

causal         paso  100 | loss 3.2908


causal         paso  200 | loss 3.0997


causal         paso  300 | loss 2.5964


causal         paso  400 | loss 2.4790


causal         paso  500 | loss 2.3497


...

bidireccional  paso  500 | loss 0.2072


bidireccional  paso  600 | loss 0.0822


bidireccional  paso  700 | loss 0.0476


causal         train 2.1815 | val 2.1981


bidireccional  train 0.0482 | val 0.0495


## Generación autoregresiva

In [6]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=200):
    """Muestreo carácter a carácter: en generación solo existe el pasado, el futuro aún no se escribió."""
    model.eval()
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        logits, _ = model(torch.tensor([ids[-block_size:]], device=device))
        probs = torch.softmax(logits[0, -1], dim=-1)
        ids.append(int(torch.multinomial(probs, 1)))
    return decode(ids)


for name, m in models.items():
    torch.manual_seed(SEED)
    print(f"=== {name} ===")
    print(generate(m, "ROMEO:"))
    print()

=== causal ===


ROMEO:

Cllar tcadeneane in roling heath anons:
Ane thil po ist if wis tee; and yonsenimss
Asf and apoteancey mou ber soocesl tha'l promg?

KIG IUES:
Yopor hit noninct deanche
Wk toutath; shat is thes tof t

=== bidireccional ===


ROMEO:::ousssssssseeeeeeeeeeeeeeuuuueeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeyyyyyyyyyyyyyyyyyyyeeeeeeeoeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeaeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeoIee



# Respuestas

**a. La máscara causal de GPT**

Un modelo de lenguaje aprende a predecir cada token usando solo los anteriores: P(x₁, …, x_T) = ∏ P(x_t | x₁, …, x_{t−1}). La máscara causal garantiza eso, pone −∞ en la atención hacia las posiciones futuras, así que después del softmax esos pesos valen 0. Sin ella, la posición t podría ver x_{t+1}, que es justo lo que tiene que predecir, y el modelo aprendería a copiar en lugar de predecir.

Lo comprobamos entrenando el Mini-GPT con y sin máscara, 700 pasos con las mismas condiciones. Los dos avanzan igual hasta el paso 400. Después, el modelo sin máscara descubre que puede copiar y su loss de validación cae a 0.05 (perplejidad de 1.05, contra 9.01 del causal). Parece un modelo perfecto, pero al generar escribe "ROMEO:::ousssssseeeee…", porque al generar el siguiente carácter todavía no existe y ya no tiene nada que copiar.

Por eso la máscara es necesaria: hace que el entrenamiento se parezca a la generación. BERT sí usa atención bidireccional, pero evita este problema escondiendo con [MASK] las palabras que tiene que predecir.

**b. Preentrenar y adaptar**

GPT solo ve el pasado. Eso le permite generar texto, pero limita cómo entiende cada palabra. BERT ve la oración completa, así que entiende mejor el contexto, pero no puede generar de izquierda a derecha.

El Task 3 lo muestra bien. "El banco central…" y "El banco de peces…" empiezan igual, así que un modelo causal le daría a "banco" exactamente el mismo vector en las dos, porque lo que define el significado viene después de la palabra. BERT sí lo ve y las distingue (similitud de 0.66), mientras que Word2Vec siempre da 1.0.

El paradigma de preentrenar y adaptar resuelve esta tensión. Cada modelo se preentrena una sola vez con mucho texto sin etiquetas, y después se escoge el que conviene para la tarea: GPT para generar y BERT para entender (clasificar, desambiguar, etc.). Luego se adapta con un poco de fine-tuning o se usan directamente sus embeddings.